# Notebook 2: Joins and Dataset Assembly

This notebook assembles the final analytical datasets from the raw and intermediate data produced in Notebook 1. We perform two main pieces of work:

1. **Build the crashes-for-analysis dataset** — one row per crash, filtered to 2018-2025, with derived flags (`severe_crash`, `has_ped_cyc`) and community area assignment attached.
2. **Build the community area demographics table** — 77 rows, with tract-level Census data aggregated up to the community area level using appropriate aggregation rules (sum for counts, population-weighted mean for income, sum-numerator-over-sum-denominator for percentages).


### Inputs (from Notebook 1)
- `crashes.csv` — 1,046,569 raw crashes
- `people.csv` — 2,297,078 person records
- `census.csv` — 1,328 Cook County tracts with 6 demographic variables
- `community_areas.geojson` — 77 Chicago community area polygons
- TIGER/Line tract shapefile (re-downloaded as needed)

### Outputs
- `crashes_for_analysis.csv` — one row per crash in the 2018-2025 window with all labels attached
- `community_area_demographics.csv` — one row per community area with aggregated demographics

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd

## Part A: Load Inputs from Notebook 1

In [ ]:
crashes_df = pd.read_csv('crashes.csv', low_memory=False)
people_df = pd.read_csv('people.csv', low_memory=False)
vehicles_df = pd.read_csv('vehicles.csv', low_memory=False)
census_df = pd.read_csv('census.csv')
community_areas = gpd.read_file('community_areas.geojson')

In [ ]:
print(crashes_df.shape, people_df.shape, vehicles_df.shape, census_df.shape, community_areas.shape)

(1046569, 48) (2297078, 29) (2133834, 71) (1328, 12) (77, 6)


## Part B: Filter Crashes to Analysis Window

Per Notebook 1's EDA, citywide crash data coverage is consistent only from 2018 onward (the city's E-Crash system rolled out citywide in September 2017). We filter to **2018-2025**, dropping pre-2018 records (incomplete coverage) and 2026 records (only 3.5 months of partial data, which would distort year-over-year analyses).

In [ ]:
crashes_df['CRASH_DATE'].dtype

dtype('O')

`CRASH_DATE` is loaded as a string from CSV. We convert it to a proper datetime and extract the year for filtering.

In [ ]:
crashes_df['CRASH_DATE'] = pd.to_datetime(crashes_df['CRASH_DATE'])
crashes_df['crash_year'] = crashes_df['CRASH_DATE'].dt.year
crashes_df['crash_year'].value_counts().sort_index()

crash_year
2013         2
2014         6
2015      9834
2016     44297
2017     83786
2018    118952
2019    117764
2020     92096
2021    108767
2022    108412
2023    110755
2024    112055
2025    109111
2026     30732
Name: count, dtype: int64

We filter to the 2018-2025 window and check the number of rows dropped on each side.

In [ ]:
n_before = len(crashes_df)
n_pre_2018 = (crashes_df['crash_year'] < 2018).sum()
n_post_2025 = (crashes_df['crash_year'] > 2025).sum()

crashes_df = crashes_df[crashes_df['crash_year'].between(2018, 2025)].reset_index(drop=True)

print(f"Before filter:    {n_before:,}")
print(f"Dropped pre-2018: {n_pre_2018:,}")
print(f"Dropped 2026:     {n_post_2025:,}")
print(f"After filter:     {len(crashes_df):,}")

Before filter:    1,046,569
Dropped pre-2018: 137,925
Dropped 2026:     30,732
After filter:     877,912


## Part C: Drop Crashes with Invalid Coordinates

Spatial joining requires valid `LATITUDE` and `LONGITUDE`. We drop crashes with null coordinates and crashes whose coordinates fall outside Chicago's bounding box (likely typos).

Number of crashes with null coordinates

In [ ]:
null_coords = crashes_df['LATITUDE'].isna() | crashes_df['LONGITUDE'].isna()
null_coords.sum()

np.int64(7151)

Next, we look at crashes with non-null coordinates that fall outside Chicago's bounding box. These are likely data entry errors (e.g., a misplaced decimal).

In [ ]:
in_bounds = (
    crashes_df['LATITUDE'].between(41.6, 42.1) &
    crashes_df['LONGITUDE'].between(-87.95, -87.5)
)

out_of_bounds = crashes_df[(~null_coords) & (~in_bounds)]
out_of_bounds[['CRASH_DATE', 'LATITUDE', 'LONGITUDE', 'STREET_NAME']]

,CRASH_DATE,LATITUDE,LONGITUDE,STREET_NAME
15624,2025-11-06 08:30:00,0.0,0.0,DAMEN AVE
26332,2025-09-30 15:35:00,0.0,0.0,DAMEN AVE
29588,2025-09-19 14:10:00,0.0,0.0,DAMEN AVE
29929,2025-09-18 12:00:00,0.0,0.0,DAMEN AVE
32430,2025-09-10 10:30:00,0.0,0.0,DAMEN AVE
36467,2025-08-28 13:20:00,0.0,0.0,DAMEN AVE
46783,2025-07-27 20:48:00,0.0,0.0,DAMEN AVE
56095,2025-06-28 14:07:00,0.0,0.0,DAMEN AVE
77842,2025-04-23 14:30:00,0.0,0.0,DAMEN AVE
90355,2025-03-11 00:23:00,0.0,0.0,DAMEN AVE


We drop both null-coordinate crashes and out-of-bounds crashes (the 55 placeholder `(0,0)` rows). Both are unusable for spatial joins.

In [ ]:
n_before = len(crashes_df)

valid_coords = (~null_coords) & in_bounds
crashes_df = crashes_df[valid_coords].reset_index(drop=True)

print(f"Before:  {n_before}")
print(f"Dropped: {n_before - len(crashes_df)}")
print(f"After:   {len(crashes_df)}")

Before:  877912
Dropped: 7206
After:   870706


## Part D: Add the Severe Crash Flag

We add a boolean column `severe_crash` based on the crash-level `MOST_SEVERE_INJURY` field. A crash is severe if its most severe injury is `FATAL` or `INCAPACITATING INJURY` (the K and A categories of the standard KABCO injury scale).

In [ ]:
crashes_df['severe_crash'] = crashes_df['MOST_SEVERE_INJURY'].isin(['FATAL', 'INCAPACITATING INJURY'])
crashes_df['severe_crash'].value_counts()

severe_crash
False    854796
True      15910
Name: count, dtype: int64

## Part E: Add the Pedestrian/Cyclist Flag

The People table has one row per person involved in a crash. We compute a crash-level boolean `has_ped_cyc` by checking whether *any* person in a crash had `PERSON_TYPE` of `PEDESTRIAN` or `BICYCLE`.

This is an **aggregation, not a join**. Joining People directly onto crashes would multiply rows (one row per person per crash). Instead, we compute the flag as a one-row-per-crash result and merge it back.

In [ ]:
ped_cyc_ids = set(people_df.loc[
        people_df['PERSON_TYPE']
        .isin(['PEDESTRIAN', 'BICYCLE']),'CRASH_RECORD_ID']
)

In [ ]:
len(ped_cyc_ids)

43769

The set contains 43,769 unique crash IDs across the full People table. We apply this set against our already-filtered `crashes_df` to flag the 2018-2025 crashes that involved a pedestrian or cyclist.

In [ ]:
crashes_df['has_ped_cyc'] = crashes_df['CRASH_RECORD_ID'].isin(ped_cyc_ids)
crashes_df['has_ped_cyc'].value_counts()

has_ped_cyc
False    832522
True      38184
Name: count, dtype: int64

As a quick sanity check, we look at the intersection of the two flags. Severe crashes that also involve a pedestrian or cyclist will drive part of our equity analysis

In [ ]:
pd.crosstab(crashes_df['severe_crash'], crashes_df['has_ped_cyc'])

has_ped_cyc,False,True
severe_crash,,
False,822005,32791
True,10517,5393


We save the filtered and flagged crashes table as a checkpoint before the spatial join. The next steps add geometry and community area assignment.

In [ ]:
crashes_df.to_csv('crashes_filtered.csv', index=False)

## Part F: Spatial Join Crashes to Community Areas

We assign each crash to a community area by spatial-joining the crash points (built from `LATITUDE`, `LONGITUDE`) against the 77 community area polygons. Both inputs use the same CRS (EPSG:4326), so no reprojection is needed.

We build a GeoDataFrame from the crash coordinates, using EPSG:4326 to match the community area polygons.

In [ ]:
crashes_gdf = gpd.GeoDataFrame(
    crashes_df,
    geometry=gpd.points_from_xy(crashes_df['LONGITUDE'], crashes_df['LATITUDE']),
    crs='EPSG:4326'
)

In [ ]:
crashes_gdf

,CRASH_RECORD_ID,CRASH_DATE_EST_I,CRASH_DATE,POSTED_SPEED_LIMIT,TRAFFIC_CONTROL_DEVICE,DEVICE_CONDITION,WEATHER_CONDITION,LIGHTING_CONDITION,FIRST_CRASH_TYPE,TRAFFICWAY_TYPE,...,CRASH_HOUR,CRASH_DAY_OF_WEEK,CRASH_MONTH,LATITUDE,LONGITUDE,LOCATION,crash_year,severe_crash,has_ped_cyc,geometry
0,16d6cdf6f2cfcd2459252d7459a332eb27fd16016b42f8...,NaN,2025-12-31 23:52:00,30,STOP SIGN/FLASHER,FUNCTIONING PROPERLY,UNKNOWN,"DARKNESS, LIGHTED ROAD",REAR END,T-INTERSECTION,...,23,4,12,41.890684,-87.639395,POINT (-87.63939528337 41.890684302388),2025,False,False,POINT (-87.6394 41.89068)
1,0e8cbefa9568ba4b339cd7ada972571acc3598e1cf8038...,NaN,2025-12-31 23:50:00,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,UNKNOWN,"DARKNESS, LIGHTED ROAD",REAR END,FOUR WAY,...,23,4,12,41.809017,-87.642999,POINT (-87.642999120383 41.809017460392),2025,False,False,POINT (-87.643 41.80902)
2,972adced5c19353f317099ef86bc038a7f307a7367ee43...,NaN,2025-12-31 23:50:00,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,UNKNOWN,UNKNOWN,SIDESWIPE SAME DIRECTION,NOT DIVIDED,...,23,4,12,41.985399,-87.656597,POINT (-87.656597159091 41.985398515961),2025,False,False,POINT (-87.6566 41.9854)
3,ba24b096416b338dd74859db8c4e7cb11a5417b6633aff...,NaN,2025-12-31 23:50:00,30,NO CONTROLS,NO CONTROLS,CLEAR,"DARKNESS, LIGHTED ROAD",SIDESWIPE SAME DIRECTION,NOT DIVIDED,...,23,4,12,41.888460,-87.639826,POINT (-87.639826408303 41.888460219921),2025,False,False,POINT (-87.63983 41.88846)
4,43ab1ced932cf882616475d1406277d6962fe09c637c1d...,NaN,2025-12-31 23:48:00,30,NO CONTROLS,NO CONTROLS,UNKNOWN,"DARKNESS, LIGHTED ROAD",REAR END,DIVIDED - W/MEDIAN (NOT RAISED),...,23,4,12,41.895558,-87.624120,POINT (-87.624120063627 41.895558109626),2025,False,False,POINT (-87.62412 41.89556)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
870701,821ee3270b2c503fd0905415f674d3f4a63e1ac15d9100...,NaN,2018-01-01 00:07:00,35,NO CONTROLS,FUNCTIONING PROPERLY,SNOW,"DARKNESS, LIGHTED ROAD",PARKED MOTOR VEHICLE,NOT DIVIDED,...,0,2,1,41.683946,-87.622993,POINT (-87.622993129041 41.683945885086),2018,False,False,POINT (-87.62299 41.68395)
870702,dfb3229940c672426c4010d7348d16b69b030191213b86...,Y,2018-01-01 00:05:00,30,NO CONTROLS,NO CONTROLS,CLEAR,"DARKNESS, LIGHTED ROAD",PARKED MOTOR VEHICLE,NOT DIVIDED,...,0,2,1,41.949417,-87.668774,POINT (-87.66877431033 41.949417412921),2018,False,False,POINT (-87.66877 41.94942)
870703,0ad333cd371ec3976f9fac8160ccfc91f0399b48a92b7f...,NaN,2018-01-01 00:00:00,25,NO CONTROLS,NO CONTROLS,UNKNOWN,UNKNOWN,PARKED MOTOR VEHICLE,NOT DIVIDED,...,0,2,1,41.681735,-87.641014,POINT (-87.641013916869 41.681735373094),2018,False,False,POINT (-87.64101 41.68174)
870704,351626f25a540a0fccbe42fb815b14ae6c75c15676bf7c...,NaN,2018-01-01 00:00:00,30,NO CONTROLS,NO CONTROLS,CLEAR,UNKNOWN,PARKED MOTOR VEHICLE,ONE-WAY,...,0,2,1,41.955842,-87.650268,POINT (-87.65026813462 41.955842185147),2018,False,False,POINT (-87.65027 41.95584)


In [ ]:
crashes_gdf.shape

(870706, 52)

Both GeoDataFrames must share the same CRS for the spatial join to work correctly.

In [ ]:
print(crashes_gdf.crs)
print(community_areas.crs)

EPSG:4326
EPSG:4326


We perform a left spatial join: each crash point gets matched to the community area polygon that contains it. The `predicate='within'` ensures crashes are matched only if their point falls inside a polygon. We use a left join so unmatched crashes (if any) are preserved for inspection.

In [ ]:
crashes_joined = gpd.sjoin(
    crashes_gdf,
    community_areas[['community', 'area_numbe', 'geometry']],
    how='left',
    predicate='within'
)

In [ ]:
crashes_joined

,CRASH_RECORD_ID,CRASH_DATE_EST_I,CRASH_DATE,POSTED_SPEED_LIMIT,TRAFFIC_CONTROL_DEVICE,DEVICE_CONDITION,WEATHER_CONDITION,LIGHTING_CONDITION,FIRST_CRASH_TYPE,TRAFFICWAY_TYPE,...,LATITUDE,LONGITUDE,LOCATION,crash_year,severe_crash,has_ped_cyc,geometry,index_right,community,area_numbe
0,16d6cdf6f2cfcd2459252d7459a332eb27fd16016b42f8...,NaN,2025-12-31 23:52:00,30,STOP SIGN/FLASHER,FUNCTIONING PROPERLY,UNKNOWN,"DARKNESS, LIGHTED ROAD",REAR END,T-INTERSECTION,...,41.890684,-87.639395,POINT (-87.63939528337 41.890684302388),2025,False,False,POINT (-87.6394 41.89068),7.0,NEAR NORTH SIDE,8
1,0e8cbefa9568ba4b339cd7ada972571acc3598e1cf8038...,NaN,2025-12-31 23:50:00,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,UNKNOWN,"DARKNESS, LIGHTED ROAD",REAR END,FOUR WAY,...,41.809017,-87.642999,POINT (-87.642999120383 41.809017460392),2025,False,False,POINT (-87.643 41.80902),60.0,NEW CITY,61
2,972adced5c19353f317099ef86bc038a7f307a7367ee43...,NaN,2025-12-31 23:50:00,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,UNKNOWN,UNKNOWN,SIDESWIPE SAME DIRECTION,NOT DIVIDED,...,41.985399,-87.656597,POINT (-87.656597159091 41.985398515961),2025,False,False,POINT (-87.6566 41.9854),76.0,EDGEWATER,77
3,ba24b096416b338dd74859db8c4e7cb11a5417b6633aff...,NaN,2025-12-31 23:50:00,30,NO CONTROLS,NO CONTROLS,CLEAR,"DARKNESS, LIGHTED ROAD",SIDESWIPE SAME DIRECTION,NOT DIVIDED,...,41.888460,-87.639826,POINT (-87.639826408303 41.888460219921),2025,False,False,POINT (-87.63983 41.88846),27.0,NEAR WEST SIDE,28
4,43ab1ced932cf882616475d1406277d6962fe09c637c1d...,NaN,2025-12-31 23:48:00,30,NO CONTROLS,NO CONTROLS,UNKNOWN,"DARKNESS, LIGHTED ROAD",REAR END,DIVIDED - W/MEDIAN (NOT RAISED),...,41.895558,-87.624120,POINT (-87.624120063627 41.895558109626),2025,False,False,POINT (-87.62412 41.89556),7.0,NEAR NORTH SIDE,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
870701,821ee3270b2c503fd0905415f674d3f4a63e1ac15d9100...,NaN,2018-01-01 00:07:00,35,NO CONTROLS,FUNCTIONING PROPERLY,SNOW,"DARKNESS, LIGHTED ROAD",PARKED MOTOR VEHICLE,NOT DIVIDED,...,41.683946,-87.622993,POINT (-87.622993129041 41.683945885086),2018,False,False,POINT (-87.62299 41.68395),52.0,WEST PULLMAN,53
870702,dfb3229940c672426c4010d7348d16b69b030191213b86...,Y,2018-01-01 00:05:00,30,NO CONTROLS,NO CONTROLS,CLEAR,"DARKNESS, LIGHTED ROAD",PARKED MOTOR VEHICLE,NOT DIVIDED,...,41.949417,-87.668774,POINT (-87.66877431033 41.949417412921),2018,False,False,POINT (-87.66877 41.94942),5.0,LAKE VIEW,6
870703,0ad333cd371ec3976f9fac8160ccfc91f0399b48a92b7f...,NaN,2018-01-01 00:00:00,25,NO CONTROLS,NO CONTROLS,UNKNOWN,UNKNOWN,PARKED MOTOR VEHICLE,NOT DIVIDED,...,41.681735,-87.641014,POINT (-87.641013916869 41.681735373094),2018,False,False,POINT (-87.64101 41.68174),52.0,WEST PULLMAN,53
870704,351626f25a540a0fccbe42fb815b14ae6c75c15676bf7c...,NaN,2018-01-01 00:00:00,30,NO CONTROLS,NO CONTROLS,CLEAR,UNKNOWN,PARKED MOTOR VEHICLE,ONE-WAY,...,41.955842,-87.650268,POINT (-87.65026813462 41.955842185147),2018,False,False,POINT (-87.65027 41.95584),2.0,UPTOWN,3


In [ ]:
crashes_joined.shape

(870706, 55)

We check for crashes that didn't match a community area. Any unmatched rows would indicate crashes whose coordinates fall just outside the city boundary.

In [ ]:
crashes_joined['community'].isna().sum()

np.int64(362)

362 crashes (about 0.04%) didn't match a community area. We inspect them to confirm they're edge cases at the city boundary.

In [ ]:
unmatched = crashes_joined[crashes_joined['community'].isna()]
unmatched[['CRASH_DATE', 'LATITUDE', 'LONGITUDE', 'STREET_NAME']].head(10)

,CRASH_DATE,LATITUDE,LONGITUDE,STREET_NAME
32,2025-12-31 22:16:00,41.892338,-87.604045,GRAND AVE
3836,2025-12-16 16:55:00,41.683828,-87.721783,115TH ST
5431,2025-12-11 07:50:00,41.742010,-87.741109,83RD ST
6936,2025-12-06 12:00:00,41.986383,-87.836434,CUMBERLAND AVE
8781,2025-11-29 17:29:00,41.962601,-87.880743,MONTROSE AVE
9590,2025-11-26 17:30:00,41.742010,-87.741109,83RD ST
12230,2025-11-16 19:40:00,41.909143,-87.775639,NORTH AVE
12243,2025-11-16 18:25:00,41.691760,-87.693733,111TH ST
19088,2025-10-24 15:47:00,41.986383,-87.836434,CUMBERLAND AVE
20113,2025-10-21 06:30:00,41.775935,-87.575036,COAST GUARD DR


The unmatched crashes are at the city's edges (lakefront, O'Hare boundary, southern city limit). We drop them since they cannot be assigned to a single community area without ambiguity.

In [ ]:
n_before = len(crashes_joined)
crashes_joined = crashes_joined.dropna(subset=['community']).reset_index(drop=True)
print(f"Before: {n_before:,}")
print(f"After:  {len(crashes_joined):,}")
print(f"Dropped: {n_before - len(crashes_joined):,}")

Before: 870,706
After:  870,344
Dropped: 362


We do a final sanity check that all 77 community areas appear in the joined data. If any are missing, the spatial join may have failed for that area.

In [ ]:
crashes_joined['community'].nunique()

77

We look at the distribution of crashes per community area as a final sanity check. The numbers should vary widely (small downtown areas vs. large outlying ones) but no community area should be unreasonably empty.

In [ ]:
crashes_joined['community'].value_counts().describe()

count       77.000000
mean     11303.168831
std       8199.455491
min        761.000000
25%       5875.000000
50%      10007.000000
75%      14951.000000
max      41419.000000
Name: count, dtype: float64

Before saving, we drop the `geometry` column (not needed for analysis, since we have `LATITUDE` and `LONGITUDE`) and the `index_right` column from the spatial join. All other Crash columns are preserved for use in later analysis.

In [ ]:
crashes_for_analysis = crashes_joined.drop(columns=['geometry', 'index_right'])
crashes_for_analysis.shape

(870344, 53)

We save the final crashes-for-analysis dataset.

In [ ]:
crashes_for_analysis.to_csv('crashes_for_analysis.csv', index=False)

### Part F Summary

We assembled the analysis dataset by:
- Filtering 1,046,569 raw crashes to the 2018-2025 window (dropped 168,657)
- Removing 7,206 crashes with invalid coordinates
- Adding the `severe_crash` flag (15,910 severe crashes, 1.83% of total)
- Adding the `has_ped_cyc` flag (38,184 ped/cyc-involved crashes, 4.39% of total)
- Spatial-joining each crash to its community area, dropping 362 unmatched edge cases

Final analysis dataset: **870,344 rows × 53 columns**, saved as `crashes_for_analysis.csv`. All 77 community areas are represented.

## Part G: Aggregate Demographics to Community Areas

The Census ACS data is at the tract level (1,328 tracts), but our analysis aggregates to the community area level (77 areas). To bridge this, we:

1. Build a tract-to-community-area crosswalk by spatially joining tract polygons to community area polygons
2. Aggregate tract-level demographics up to community areas using appropriate aggregation rules:
   - **Counts** (population, households): summed across tracts
   - **Median income**: population-weighted mean (small high-income tracts shouldn't get equal weight to large low-income tracts)
   - **Percentages** (% POC, % zero-vehicle households): recomputed from the underlying counts (sum of numerators divided by sum of denominators)

The result is a 77-row demographics table that we'll join onto crashes by community area.

We reload the TIGER/Line tract shapefile (downloaded in Notebook 1 for the original tract spatial join) and filter to Cook County.

In [ ]:
tracts_url = "https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_17_tract.zip"
tracts_gdf = gpd.read_file(tracts_url)
cook_tracts = tracts_gdf[tracts_gdf['COUNTYFP'] == '031'].reset_index(drop=True)
cook_tracts.shape

(1332, 14)

We confirm the tract shapefile and community area shapefile share a CRS for the spatial join.

In [ ]:
print(cook_tracts.crs)
print(community_areas.crs)

EPSG:4269
EPSG:4326


The two shapefiles use different CRS values (TIGER files use NAD83, the Chicago Data Portal uses WGS 84). We reproject the tracts to EPSG:4326 to match the community areas.

In [ ]:
cook_tracts = cook_tracts.to_crs('EPSG:4326')
cook_tracts.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

We build the crosswalk by spatial-joining tract polygons against community area polygons. Since tracts can technically straddle community area boundaries, we use the tract's centroid (rather than its full polygon) to assign each tract to a single community area. This avoids one tract being matched to multiple communities.

In [ ]:
tract_centroids = cook_tracts.to_crs('EPSG:3435').copy()
tract_centroids['geometry'] = tract_centroids.geometry.centroid
tract_centroids = tract_centroids.to_crs('EPSG:4326')

crosswalk = gpd.sjoin(
    tract_centroids[['GEOID', 'TRACTCE', 'geometry']],
    community_areas[['community', 'area_numbe', 'geometry']],
    how='left',
    predicate='within'
)
crosswalk.shape

(1332, 6)

We check whether any tracts failed to match a community area. The Cook County tract universe (1,332) is larger than Chicago's, so we expect a meaningful share of tracts in suburban Cook County to be unmatched.

In [ ]:
crosswalk['community'].isna().sum()

np.int64(551)

In [ ]:
crosswalk = crosswalk.dropna(subset=['community']).reset_index(drop=True)
crosswalk.shape

(781, 6)

We merge the tract-level Census demographics onto the crosswalk so each tract row has both its community area assignment and its demographic values. The join key is `TRACTCE`, the 6-character tract code. We use `census.csv`'s `tract` column, which we'll first convert to the same string format we used in Notebook 1.

In [ ]:
census_df['tract'] = census_df['tract'].astype(str).str.zfill(6)
crosswalk_demo = crosswalk.merge(
    census_df,
    left_on='TRACTCE',
    right_on='tract',
    how='left'
)
crosswalk_demo.shape

(781, 18)

We check that every Chicago tract matched to a Census record. Any nulls in `total_population` would indicate a tract that's in our crosswalk but wasn't in our cleaned `census.csv` (likely one of the 4 zero-population tracts we dropped during EDA in Notebook 1).

In [ ]:
crosswalk_demo['total_population'].isna().sum()

np.int64(3)

The 3 unmatched tracts are zero-population tracts inside Chicago (parks, airport, or institutional areas) that we dropped in Notebook 1's EDA. We drop them from the crosswalk before aggregating since they have no demographics to contribute.

In [ ]:
crosswalk_demo = crosswalk_demo.dropna(subset=['total_population']).reset_index(drop=True)
crosswalk_demo.shape

(778, 18)

We aggregate the tract-level demographics up to the community area level. Different variables use different aggregation rules:

- **Total population, total households**: summed across the community area's tracts
- **Race universe and non-Hispanic white populations, zero-vehicle households**: summed (used to recompute percentages from raw counts)
- **Median household income**: population-weighted mean across tracts (excluding the 15 tracts where median income is null)

The percentage variables (`pct_poc`, `pct_zero_vehicle_hh`) will be derived from the summed numerators and denominators in a separate step, since aggregating percentages directly is incorrect.

In [ ]:
def weighted_income(group):
    valid = group.dropna(subset=['median_household_income'])
    if valid['total_population'].sum() == 0:
        return np.nan
    return (valid['median_household_income'] * valid['total_population']).sum() / valid['total_population'].sum()

ca_demographics = crosswalk_demo.groupby(['community', 'area_numbe']).apply(
    lambda g: pd.Series({
        'total_population': g['total_population'].sum(),
        'total_households': g['total_households'].sum(),
        'nh_white_population': g['nh_white_population'].sum(),
        'race_universe_population': g['race_universe_population'].sum(),
        'zero_vehicle_households': g['zero_vehicle_households'].sum(),
        'pop_weighted_income': weighted_income(g),
        'n_tracts': len(g),
    }),
    include_groups=False
).reset_index()

ca_demographics.shape

(77, 9)

In [ ]:
ca_demographics

,community,area_numbe,total_population,total_households,nh_white_population,race_universe_population,zero_vehicle_households,pop_weighted_income,n_tracts
0,ALBANY PARK,14,46620.0,17294.0,15530.0,46620.0,3256.0,83446.178743,11.0
1,ARCHER HEIGHTS,57,14021.0,4610.0,1706.0,14021.0,1153.0,66848.541000,5.0
2,ARMOUR SQUARE,34,14207.0,5833.0,2206.0,14207.0,2185.0,43710.278243,5.0
3,ASHBURN,70,42079.0,13183.0,3577.0,42079.0,1162.0,83262.282801,8.0
4,AUBURN GRESHAM,71,45049.0,17752.0,374.0,45049.0,4977.0,47639.120202,15.0
...,...,...,...,...,...,...,...,...,...
72,WEST LAWN,65,32649.0,9702.0,3157.0,32649.0,1042.0,72994.766486,6.0
73,WEST PULLMAN,53,24470.0,9468.0,296.0,24470.0,2024.0,49911.154230,8.0
74,WEST RIDGE,2,78227.0,26301.0,30298.0,78227.0,3683.0,73520.787950,14.0
75,WEST TOWN,24,82051.0,39908.0,51727.0,82051.0,8211.0,137581.742831,30.0


We compute the two derived percentage columns from the summed counts (the correct way to aggregate percentages from underlying tract data), then assign each community area to an income quartile based on `pop_weighted_income`.

In [ ]:
ca_demographics['pct_poc'] = 1 - (ca_demographics['nh_white_population'] / ca_demographics['race_universe_population'])
ca_demographics['pct_zero_vehicle_hh'] = ca_demographics['zero_vehicle_households'] / ca_demographics['total_households']
ca_demographics['income_quartile'] = pd.qcut(ca_demographics['pop_weighted_income'], q=4, labels=[1, 2, 3, 4])

ca_demographics[['pct_poc', 'pct_zero_vehicle_hh', 'pop_weighted_income', 'income_quartile']].describe()

,pct_poc,pct_zero_vehicle_hh,pop_weighted_income
count,77.000000,77.000000,77.000000
mean,0.734414,0.240314,72850.444669
std,0.253875,0.128200,31459.564058
min,0.186277,0.039933,19009.473856
25%,0.516159,0.131802,49911.154230
50%,0.866489,0.231774,67970.925588
75%,0.952709,0.347396,90001.469200
max,1.000000,0.575397,158406.589981


We confirm the income quartile assignments are roughly evenly distributed.

In [ ]:
ca_demographics['income_quartile'].value_counts().sort_index()

income_quartile
1    20
2    19
3    19
4    19
Name: count, dtype: int64

We save the community area demographics table.

In [ ]:
ca_demographics.to_csv('community_area_demographics.csv', index=False)
ca_demographics.shape

(77, 12)

### Part G Summary

We aggregated tract-level Census demographics up to Chicago's 77 community areas:
- Built a tract-to-community-area crosswalk by spatially joining tract centroids against community area polygons (1,332 Cook County tracts → 778 Chicago tracts after dropping suburban tracts and zero-population tracts)
- Aggregated counts by summing across each community area's tracts
- Computed median income as a population-weighted mean of tract medians
- Recomputed percentages from summed numerators and denominators (the correct approach for aggregating proportions)
- Assigned each community area to an income quartile

Final demographics table: **77 rows × 12 columns**, saved as `community_area_demographics.csv`.

## Notebook 2 Summary

We assembled two analytical datasets from the inputs produced in Notebook 1:

**`crashes_for_analysis.csv`** (870,344 rows × 53 columns)
Each row is one crash in the 2018-2025 window with:
- Original Crash table fields preserved
- `severe_crash` flag (True for fatal or incapacitating injury)
- `has_ped_cyc` flag (True if any involved person was a pedestrian or bicyclist)
- `community` and `area_numbe` from the spatial join against community area polygons

**`community_area_demographics.csv`** (77 rows × 12 columns)
Each row is one Chicago community area with:
- Aggregated population, household, and race counts
- Population-weighted median household income
- Derived `pct_poc` and `pct_zero_vehicle_hh`
- Income quartile assignment (1 = lowest, 4 = highest)

These two tables join on `community` (or `area_numbe`) and form the foundation for the analysis in Notebook 3.